<a href="https://colab.research.google.com/github/Shambo25/flyrankAIinternshipStarter/blob/main/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shambo25/flyrankAIinternshipStarter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Lane:** Refresh / Content Opportunity Scoring.

**Task type:** ranking / scoring, not a yes/no classifier in production.

The decision is *which page an editor should review first* when they only have time for a short queue. That maps to a priority score and a ranked list. Classification is only a helper: we can train a decline-risk score, then **rank** by it. Clustering is the wrong output for this decision, and a single class label does not tell anyone what to open first.

**One-paragraph frame:** For a content editor (or account operator), deciding which existing pages to review first, we will build a ranked review queue from the starter 30k-page snapshot (later: warehouse daily facts), scoring refresh priority. On the starter slice the evaluation proxy is `trend_direction == "down"` (a **defined rule**, not a future outcome). Success is **precision@50** on a client-holdout split: of the 50 pages we put at the top, how many match the proxy — and later, a future-window decline we define ourselves. A wrong call costs wasted editor hours on low-value pages, or a visible page left decaying. A plain if-statement is not enough because volume, age, position, CTR, and engagement interact, missingness follows `content_type`, and a single threshold either catches almost nobody or almost everybody. We will claim only **observed / directional / decision-support** results — not that a refresh caused recovery, and not that we predicted Google.

**Output → real content action:** the deliverable is a ranked queue, not a class label. For each page we attach one suggested action a human can take this week — `refresh` (update a visible, aging, or declining page), `expand_and_refresh` (thin but visible copy), `refresh_and_review_ctr` (page is seen but CTR is weak), or `monitor` (do not spend hours yet). Nobody auto-publishes. The editor opens the page, checks the reason codes, and decides.

In [14]:
from pathlib import Path

import pandas as pd



DATA_PATH = Path("content_refresh_anonymized.csv")

df = pd.read_csv(DATA_PATH)
print("loaded:", DATA_PATH.resolve())
print("rows x cols:", df.shape)
print("unique pages (content_id):", df["content_id"].nunique())
print("unique clients (client_id):", df["client_id"].nunique())
print("task_type_for_this_lane: ranking/scoring")


loaded: /content/content_refresh_anonymized.csv
rows x cols: (30000, 44)
unique pages (content_id): 30000
unique clients (client_id): 32
task_type_for_this_lane: ranking/scoring


## 2. Target or proxy

**What we score:** a page's priority for human refresh review.

**Starter proxy (this notebook):** `is_declining_label = (trend_direction == "down")`. That bucket is a **defined rule** on the same 90-day snapshot: last-30d vs prev-30d impressions, with `down` meaning a drop worse than −20%. It is **not** an observed later outcome. The dictionary is explicit: `trend_direction` and `trend_pct` must **never** be features, or the model just relearns the rule.

**Capstone target (warehouse, later):** an **observed** future-window label, e.g. features from a prior 90 days → impressions (or clicks) decline over the next 30 days, with a volume floor, and June 2026 kept as a sealed test month. That label is measured after the decision point. This starter proxy only lets us practice the workflow until that contract exists.

We do not rebuild FlyRank product flags (`health_score`, `priority_score`, …) as a target — they are not in the data, and predicting a reconstructed rule would be circular.

In [15]:
# Proxy is a rule on trend_direction, which itself is a rule on trend_pct.
# That is defined, not observed-in-the-future.
df["is_declining_proxy"] = (df["trend_direction"].str.lower() == "down").astype(int)

print("proxy positive rate:", round(df["is_declining_proxy"].mean(), 4))
print("proxy positives:", int(df["is_declining_proxy"].sum()), "of", len(df))
print("trend_direction counts:")
print(df["trend_direction"].value_counts(dropna=False).to_string())
print("trend_pct missing (prev-30d impressions were 0):", int(df["trend_pct"].isna().sum()))
print("never_use_as_features:", ["trend_direction", "trend_pct", "content_id", "client_id"])


proxy positive rate: 0.5421
proxy positives: 16262 of 30000
trend_direction counts:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
trend_pct missing (prev-30d impressions were 0): 3388
never_use_as_features: ['trend_direction', 'trend_pct', 'content_id', 'client_id']


## 3. Success metric

**Primary metric: precision@50.** Editors review a short queue. The honest question is: of the 50 pages we put first, how many actually match the label we defined? That number is computable today on a baseline, before any new model. The naive ranker below is scored on the **full 30k slice** so we can compute *a* precision@50 today; the 0.240 / 0.740 numbers are the reference pipeline on a **client-holdout** split — do not mix those two setups.

**What 'good' means on this starter proxy (directional, not a warehouse claim):**
- Random ranking should sit near the base rate (~0.54 of pages are `down`).
- The committed reference pipeline, on this same 30k slice with **client-holdout**, measured precision@50 of **0.24** for the hand-rule baseline and **0.74** for random forest — against the *proxy* label. Beating the hand rule on that split is the bar for 'the ranker is useful as decision-support on this slice.'
- We do **not** treat 0.74 as proof of future decline, recovery, or a Google algorithm.

ROC-AUC is a secondary diagnostic (ranking quality across all thresholds). We will not pick a metric after seeing test numbers.

In [16]:
import numpy as np

K = 50
base_rate = df["is_declining_proxy"].mean()

# Naive non-leaky ranker: older + more visible first. No trend_* columns.
naive_score = df["days_since_last_update"] * np.log1p(df["impressions_90d"].clip(lower=0))
topk = df.assign(naive_score=naive_score).nlargest(K, "naive_score")
prec_at_k = topk["is_declining_proxy"].mean()

print(f"base_rate (random precision@{K} in expectation): {base_rate:.4f}")
print(
    f"naive_ranker precision@{K} on FULL slice (not holdout): {prec_at_k:.4f}  "
    f"({int(topk['is_declining_proxy'].sum())}/{K} proxy-positive)"
)
print("reference_pipeline_on_this_slice (client-holdout, proxy label):")
print("  baseline_rules precision@50 = 0.240")
print("  random_forest  precision@50 = 0.740")
print("good_bar_this_week: beat 0.240 on the same proxy + split; do not read 0.740 as a future-outcome result")


base_rate (random precision@50 in expectation): 0.5421
naive_ranker precision@50 on FULL slice (not holdout): 0.6400  (32/50 proxy-positive)
reference_pipeline_on_this_slice (client-holdout, proxy label):
  baseline_rules precision@50 = 0.240
  random_forest  precision@50 = 0.740
good_bar_this_week: beat 0.240 on the same proxy + split; do not read 0.740 as a future-outcome result


## 4. The unit of analysis, as a real dataframe

**One row = one content item (page)** for one pseudonymized client, with metrics over a trailing 90-day window. `content_id` is unique in this slice (30,000 pages, 32 clients). IDs are for grouping and splits only — never features.

The **decision grain** is the same: score each page, then return a ranked queue. Later warehouse work uses daily facts (`report_date × client × content`) that we will aggregate up to this page grain with explicit feature vs label windows. `avg_position == 0` is missing position (1,205 rows), not rank zero. Rates are ×100 percentages (`ctr = 0.76` means 0.76%).

In [17]:
from pathlib import Path

import pandas as pd

DATA_PATH = Path("content_refresh_anonymized.csv")

# Load the starter slice here (this section's job: show the grain).
df = pd.read_csv(DATA_PATH)
df["is_declining_proxy"] = (df["trend_direction"].str.lower() == "down").astype(int)

print("loaded:", DATA_PATH.resolve())
print("rows x cols:", df.shape)
print("one_row_equals: one content item (page) for one client, trailing-90-day snapshot")
print("unique content_id (pages):", df["content_id"].nunique())
print("unique client_id:", df["client_id"].nunique())

# Grain probe: GROUP BY content_id HAVING COUNT(*) > 1 should be empty.
dupes = df.groupby("content_id").size()
dupe_groups = int((dupes > 1).sum())
print("duplicate content_id groups (expect 0):", dupe_groups)
print("avg_position==0 (missing position, not rank zero):", int((df["avg_position"] == 0).sum()))
print(
    "measurable_opportunity (impressions_90d>=100 and sessions_90d>0):",
    int(((df["impressions_90d"] >= 100) & (df["sessions_90d"] > 0)).sum()),
)
print("ids are grouping keys only — never model features")

show_cols = [
    "content_id",
    "client_id",
    "content_type",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "days_since_last_update",
    "trend_direction",
    "is_declining_proxy",
]
# Public-safe preview: pseudonym ids only; this file has no URLs or queries.
print("\npreview (unit of analysis = one page):")
print(df[show_cols].head(5).to_string(index=False))


loaded: /content/content_refresh_anonymized.csv
rows x cols: (30000, 45)
one_row_equals: one content item (page) for one client, trailing-90-day snapshot
unique content_id (pages): 30000
unique client_id: 32
duplicate content_id groups (expect 0): 0
avg_position==0 (missing position, not rank zero): 1205
measurable_opportunity (impressions_90d>=100 and sessions_90d>0): 22006
ids are grouping keys only — never model features

preview (unit of analysis = one page):
          content_id         client_id    content_type  impressions_90d  clicks_90d  ctr  avg_position  days_since_last_update trend_direction  is_declining_proxy
content_304f48230142 client_f369cb89fc keyword article             3803          29 0.76          10.6                      20            down                   1
content_a1fb4e703a9e client_4e07408562 keyword article            15320           7 0.05          20.3                      25            down                   1
content_9aa793d4d895 client_7f2253d7e2 keyw

## 5. Why ML beats a fixed rule here

A single if-statement fails the capacity problem: the same starter rules that the reference baseline uses either fire on **17** pages (`stale_visible_page`) or on **9,759** (`low_ctr_visible_page`). Those two groups barely overlap, and a third rule (`thin_visible_page`) hits 82 pages at about the base rate. An editor cannot use 'if CTR is low' as a queue.

The pattern is messy in the data, not just in theory:
- **Many signals, tangled:** age, freshness, position (with zeros meaning missing), CTR (a ×100 percentage), volume, engagement, content type.
- **Missingness follows `content_type`:** feedly articles have ~100% missing keyword fields; keyword articles have ~28% missing `word_count`. A fillna(0) would smuggle type into the score.
- **Heavy tails and noise:** impressions start at 1; a CTR move on a tiny page is not the same as on a page with thousands of impressions.
- **The proxy itself is a threshold** on impression trend. Ranking 'who to review' needs to mix that risk with *whether the page is worth the hours* — volume × opportunity × actionability — which a fixed weight recipe can start, and a model can reweight if it beats that recipe on a held-out set of **clients**.

If a later audit shows a simple rule matching the model, we keep the rule. ML has to earn the slot.

In [18]:
stale = (df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)
thin = (df["word_count"].fillna(0) > 0) & (df["word_count"] < 1200) & (df["impressions_90d"] >= 250)
low_ctr = (
    (df["impressions_90d"] >= 500)
    & (df["avg_position"] > 0)
    & (df["avg_position"] <= 20)
    & (df["ctr"] < 0.5)
)

print("stale_visible_page n / proxy rate:", int(stale.sum()), round(df.loc[stale, "is_declining_proxy"].mean(), 4))
print("thin_visible_page n / proxy rate:", int(thin.sum()), round(df.loc[thin, "is_declining_proxy"].mean(), 4))
print("low_ctr_visible_page n / proxy rate:", int(low_ctr.sum()), round(df.loc[low_ctr, "is_declining_proxy"].mean(), 4))
print("union of those three rules:", int((stale | thin | low_ctr).sum()))
print("intersection of all three:", int((stale & thin & low_ctr).sum()))

print("\nmissingness by content_type (word_count):")
print(df.groupby("content_type")["word_count"].apply(lambda s: round(s.isna().mean(), 4)).to_string())
print("\nmissingness by content_type (search_volume):")
print(df.groupby("content_type")["search_volume"].apply(lambda s: round(s.isna().mean(), 4)).to_string())

# Map overlapping signals to one suggested action (decision-support, not auto-publish).
def suggested_action(row) -> str:
    if row["thin"]:
        return "expand_and_refresh"
    if row["low_ctr"]:
        return "refresh_and_review_ctr"
    if row["stale"]:
        return "refresh"
    return "monitor"

actions = pd.DataFrame({"stale": stale, "thin": thin, "low_ctr": low_ctr}).apply(suggested_action, axis=1)
print("\nsuggested_action counts (starter rules, this slice):")
print(actions.value_counts().to_string())
print("human_next_step: open the page, read reason codes, then refresh / expand / fix snippet / or skip")


stale_visible_page n / proxy rate: 17 0.9412
thin_visible_page n / proxy rate: 82 0.5366
low_ctr_visible_page n / proxy rate: 9759 0.6271
union of those three rules: 9830
intersection of all three: 0

missingness by content_type (word_count):
content_type
comparison article    0.000
feedly article        0.000
keyword article       0.283

missingness by content_type (search_volume):
content_type
comparison article    0.0000
feedly article        1.0000
keyword article       0.0137

suggested_action counts (starter rules, this slice):
monitor                   20170
refresh_and_review_ctr     9741
expand_and_refresh           82
refresh                       7
human_next_step: open the page, read reason codes, then refresh / expand / fix snippet / or skip


## 6. Frame check (decision, cost, claims)

**Decision this improves:** which existing page an editor opens first when they only have a short review queue.

**Who acts:** a content editor or account operator. They open the page, read reason codes, then refresh, expand, fix snippet/CTR, or skip. Nobody auto-publishes.

**Cost of a wrong call:** hours spent on a low-value page, or a visible decaying page left in the backlog. False positives waste time; false negatives miss a page that still has demand.

**Why data/ML:** a single if-statement either fires on ~17 pages or ~9,759. Volume, age, position, CTR, and missingness-by-type interact. ML earns a slot only if it beats the hand-rule baseline on a held-out set of *clients*.

**Target honesty:** this week's proxy is a **defined rule** (`trend_direction == "down"`), not a later observed outcome. The warehouse capstone target will be a future window. We will not rebuild product `health_score` / `priority_score` as a label.

**Claims we will make:** observed / directional / decision-support — "this ranking put more proxy-positive (or later-declining) pages in the top 50 than the rule baseline on this split."

**Claims we will not make:** that a refresh caused recovery; that we predicted Google; that the starter 0.74 random-forest number is a warehouse future-outcome result.

In [19]:
# Receipts for the frame — numbers already computed above, restated in one place.
print("decision: rank existing pages for human refresh review")
print("actor: editor / account operator (human in the loop)")
print("output: ranked queue + suggested_action, not auto-publish")
print("starter_proxy: defined rule  is_declining_proxy = (trend_direction == 'down')")
print("proxy_positive_rate:", round(df["is_declining_proxy"].mean(), 4))
print("unit_of_analysis: one row = one content_id (page) on a 90-day snapshot")
print("pages / clients:", df["content_id"].nunique(), "/", df["client_id"].nunique())
print("primary_metric: precision@50 (computable on a baseline today)")
print("never_features:", ["trend_direction", "trend_pct", "content_id", "client_id"])
print("claim_level: observed / directional / decision-support")
print("not_claiming: causal recovery, predicting Google, warehouse future-outcome from this slice")

decision: rank existing pages for human refresh review
actor: editor / account operator (human in the loop)
output: ranked queue + suggested_action, not auto-publish
starter_proxy: defined rule  is_declining_proxy = (trend_direction == 'down')
proxy_positive_rate: 0.5421
unit_of_analysis: one row = one content_id (page) on a 90-day snapshot
pages / clients: 30000 / 32
primary_metric: precision@50 (computable on a baseline today)
never_features: ['trend_direction', 'trend_pct', 'content_id', 'client_id']
claim_level: observed / directional / decision-support
not_claiming: causal recovery, predicting Google, warehouse future-outcome from this slice


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.